In [ ]:
# ===== Cell 1: 复用第 07 课封装的 Windows Shell 工具 =====
# shell_tool 让大模型 / Python 可以执行 dir、mkdir、findstr 等终端命令，
# 本节课用它来"发现文件"——自动扫描 math 目录里有哪些可视化页面需要扩充进知识库。
import subprocess
from langchain_core.tools import tool


@tool
def shell_tool(command: str) -> str:
    '''在 Windows 终端(cmd)中执行一条命令并返回输出结果。
    可用于查看目录(dir)、创建目录(mkdir)、查找文件内容(findstr)等。

    参数:
        command: 要执行的命令，例如 "dir /b txt\\math"
    '''
    try:
        # chcp 65001 切到 UTF-8 代码页，避免中文文件名/路径乱码
        result = subprocess.run(
            f'chcp 65001 >nul && {command}',
            shell=True, capture_output=True, text=True,
            encoding='utf-8', errors='replace', timeout=30,
        )
        if result.returncode != 0:
            return f'命令执行失败: {result.stderr.strip()}'
        return result.stdout.strip() or '(命令执行成功，无输出)'
    except subprocess.TimeoutExpired:
        return '命令执行超时（30秒），已终止。'
    except Exception as e:
        return f'执行异常: {e}'


print('shell_tool 就绪')



In [ ]:
# ===== Cell 2: 用 shell_tool 扫描 math 目录、准备知识库输出目录 =====
import os

MATH_DIR = os.path.join(os.getcwd(), 'txt', 'math')
# 生成的知识点 markdown 统一放到 math_knowledge 目录
OUT_DIR = os.path.join(os.getcwd(), 'txt', 'math_knowledge')

# 1) shell_tool 执行 dir /b：只列出文件名（不含大小日期），等价于让"终端助手"去探查目录
listing = shell_tool.invoke({'command': f'dir /b "{MATH_DIR}\\*.html"'})
html_files = [ln.strip() for ln in listing.splitlines() if ln.strip().lower().endswith('.html')]
print(f'shell_tool 扫描到 {len(html_files)} 个 HTML 可视化页面')

# 2) 输出目录不存在就创建（mkdir 在 cmd 里，存在时会有提示，无妨）
if not os.path.exists(OUT_DIR):
    shell_tool.invoke({'command': f'mkdir "{OUT_DIR}"'})
    print(f'已创建知识库目录: {OUT_DIR}')
else:
    print(f'知识库目录已存在: {OUT_DIR}')

# 预览前5个文件名（文件名本身就是知识点名）
print('\n前5个待处理文件:')
for f in html_files[:5]:
    print(' -', f)



In [ ]:
# ===== Cell 3: 解析每个 HTML 页面里的知识点元数据 =====
# 这些可视化页面的 <meta> 标签里规范地记录了学科/难度/认知动作/一句话描述，
# 是生成知识点文档的优质"原料"。
from bs4 import BeautifulSoup


def parse_html_meta(filepath):
    '''读取一个 HTML 文件，提取知识点元数据

    返回字典: name(知识点名) / grade / topic / difficulty / priority /
              cognitive(认知动作) / desc(一句话描述) / filename
    '''
    with open(filepath, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f.read(), 'html.parser')

    # title 形如 "Gram-Schmidt正交化动画 · 数学可视化"，取 · 前面的部分
    title = soup.title.string.strip() if soup.title else os.path.basename(filepath)
    name = title.split('·')[0].strip()

    # 把所有 <meta name=... content=...> 整理成字典
    meta = {}
    for tag in soup.find_all('meta'):
        key = tag.get('name')
        if key:
            meta[key] = tag.get('content', '').strip()

    # 页面主标题下面还有一行说明（p 标签），有的页面会写更详细的简介
    subtitle = ''
    p = soup.find('p')
    if p:
        subtitle = p.get_text(strip=True)

    return {
        'name': name,
        'grade': meta.get('grade', ''),
        'topic': meta.get('topic', ''),
        'difficulty': meta.get('difficulty', ''),
        'priority': meta.get('priority', ''),
        'cognitive': meta.get('cognitive', ''),
        'desc': meta.get('desc', '') or subtitle,
        'filename': os.path.basename(filepath),
    }


# 批量解析全部文件
knowledge_items = []
for fname in html_files:
    info = parse_html_meta(os.path.join(MATH_DIR, fname))
    knowledge_items.append(info)

print(f'成功解析 {len(knowledge_items)} 个知识点元数据，示例：')
example = knowledge_items[0]
for k, v in example.items():
    print(f'  {k:11s}: {v}')



In [ ]:
# ===== Cell 4: 调用付费模型，把元数据扩写成完整知识点文档 =====
# 思路：HTML 交互页面本身文字很少，但元数据(知识点名+学科+一句话描述)已点明主题；
#       让大模型基于这些线索，补充概念、原理公式、实验方法和学习要点，生成 markdown。
import time
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model='sensenova-6.8-flash-lite',
    base_url='https://token.sensenova.cn/v1',
    api_key=os.getenv('SENSENOVA_API_KEY', ''),
    temperature=0.4,
)

# 知识点文档的统一模板（告诉模型要写哪些部分、什么风格）
DOC_PROMPT = '''你是数学老师。请根据以下信息，为学生编写一份结构清晰的知识点讲解文档(Markdown格式)。

知识点：{name}
学科分类：{grade} · {topic}
难度：{difficulty}
认知动作：{cognitive}
一句话描述：{desc}
配套工具：{filename}（一个可交互操作的可视化网页）

要求：
1. 直接输出 Markdown 正文，不要寒暄、不要用代码块包裹整篇内容。
2. 文档包含以下五个部分，每部分内容充实但总字数控制在 500~700 字：
   ## 一句话理解
   ## 核心概念（这个知识点讲什么，用通俗语言解释）
   ## 原理与公式（关键定义/定理/公式，公式用文字或简单符号表达即可）
   ## 动手实验（如何操作配套的可视化网页来观察和理解它，至少写3步操作建议）
   ## 学习要点（易错点、记忆口诀或与其他知识的联系，2~4条）
3. 内容必须科学准确，围绕"{name}"展开，不要编造与该知识点无关的内容。
'''


def generate_doc(item):
    '''根据一个知识点的元数据，调用大模型生成 markdown 文档文本'''
    prompt = DOC_PROMPT.format(**item)
    # 网络抖动/限流时最多重试 3 次
    for attempt in range(3):
        try:
            resp = llm.invoke([{'role': 'user', 'content': prompt}])
            content = resp.content.strip()
            # 模型偶发返回空内容（只有表头没有正文），长度过短时也当作失败重试
            if len(content) >= 100:
                return content
            print(f'  生成「{item["name"]}」第{attempt+1}次返回内容过短({len(content)}字)，重试...')
        except Exception as e:
            print(f'  生成「{item["name"]}」第{attempt+1}次失败: {type(e).__name__}，重试...')
        time.sleep(2)
    return None


def build_markdown(item, body):
    '''在模型生成的正文前面加上统一的元信息表头'''
    header = (
        f'# {item["name"]}\n\n'
        f'> **{item["grade"]} · {item["topic"]}** ｜ '
        f'难度：{item["difficulty"]} ｜ 优先级：{item["priority"]} ｜ '
        f'认知动作：{item["cognitive"]}\n'
        f'> 配套可视化工具：`{item["filename"]}`\n\n'
        f'---\n\n'
    )
    return header + body


# 先测试 1 个，看看生成质量
test_item = knowledge_items[0]
print(f'正在为「{test_item["name"]}」生成知识点文档...')
t0 = time.time()
test_body = generate_doc(test_item)
print(f'耗时 {time.time()-t0:.1f}s，生成 {len(test_body)} 字\n')
print(build_markdown(test_item, test_body)[:800])



In [ ]:
# ===== Cell 5: 让 Agent 带着 shell 工具"自动发现并扩充"知识库 =====
# 模拟真实的知识库扩充场景：Agent 像一个资料员——
#   ① 先用 shell_tool 扫描目录，发现有哪些 HTML 页面；
#   ② 再调用 build_knowledge_tool 为指定页面生成知识点文档。
# shell_tool 是它探查文件系统的"眼睛"，build_knowledge_tool 是它写文档的"手"。
from langchain.agents import create_agent


@tool
def build_knowledge_tool(html_filename: str) -> str:
    '''为 math 目录下指定的 HTML 可视化页面生成知识点 Markdown 文档，
    并保存到 txt/math_knowledge 知识库目录。生成成功后返回保存路径。

    参数:
        html_filename: HTML 文件名（含.html后缀），例如 "Gram-Schmidt正交化动画.html"
    '''
    filepath = os.path.join(MATH_DIR, html_filename)
    if not os.path.exists(filepath):
        return f'文件不存在: {html_filename}'
    item = parse_html_meta(filepath)
    body = generate_doc(item)
    if body is None:
        return f'生成失败: {html_filename}（模型调用多次失败）'
    md = build_markdown(item, body)
    out_path = os.path.join(OUT_DIR, html_filename.replace('.html', '.md'))
    with open(out_path, 'w', encoding='utf-8') as f:
        f.write(md)
    return f'已生成知识点文档: {os.path.basename(out_path)}（{len(md)}字）'


agent = create_agent(
    model=llm,
    tools=[shell_tool, build_knowledge_tool],
    system_prompt=(
        '你是知识库扩充助手。工作流程：'
        '1) 用 shell_tool 执行 dir /b 命令扫描 math 目录，确认其中的 HTML 文件；'
        '2) 从文件列表中选择一个，调用 build_knowledge_tool 生成知识点文档；'
        '3) 向用户汇报扫描到多少个文件、为哪个文件生成了文档。'
        '注意：每次只生成一个文件作为演示，路径使用用户给出的绝对路径。'
    ),
)

result = agent.invoke({'messages': [{
    'role': 'user',
    'content': f'请扫描目录 {MATH_DIR} 下的 HTML 文件，然后任选一个生成知识点文档。'
}]})
print(result['messages'][-1].content.strip())



In [ ]:
# ===== Cell 6: 批量扩充 —— 为全部 44 个页面生成知识点文档 =====
# Agent 适合"决策与演示"，批量重复任务用普通循环更稳定、省钱。
# 已存在的 md 会自动跳过（支持断点续跑，中途失败重跑不会重复生成）。
def batch_build(items, limit=None):
    ok, skip, fail = 0, 0, []
    todo = items if limit is None else items[:limit]
    for i, item in enumerate(todo, 1):
        out_name = item['filename'].replace('.html', '.md')
        out_path = os.path.join(OUT_DIR, out_name)
        if os.path.exists(out_path):
            skip += 1
            print(f'[{i}/{len(todo)}] 跳过(已存在): {out_name}')
            continue
        body = generate_doc(item)
        if body is None:
            fail.append(item['filename'])
            print(f'[{i}/{len(todo)}] 失败: {item["filename"]}')
            continue
        with open(out_path, 'w', encoding='utf-8') as f:
            f.write(build_markdown(item, body))
        ok += 1
        print(f'[{i}/{len(todo)}] 已生成: {out_name}（{item["topic"]}）')
        time.sleep(1)   # 礼貌间隔，避免触发 API 限流
    print(f'\n完成：新生成 {ok} 个，跳过 {skip} 个，失败 {len(fail)} 个')
    if fail:
        print('失败清单:', fail)
    return fail


# limit=None 处理全部；第一次跑想先验证可以改成 limit=3 只跑前3个
fails = batch_build(knowledge_items, limit=None)



In [ ]:
# ===== Cell 7: 验收 —— 用 shell_tool 查看生成的知识库 =====
generated = shell_tool.invoke({'command': f'dir /b "{OUT_DIR}\\*.md"'})
md_list = [ln for ln in generated.splitlines() if ln.strip().endswith('.md')]
print(f'知识库中现有 {len(md_list)} 个知识点文档：')
for name in md_list[:10]:
    print(' -', name)
if len(md_list) > 10:
    print(f' ... 及其余 {len(md_list)-10} 个')

# 随机展示一个文档的前几行，确认内容质量
import random
sample_md = os.path.join(OUT_DIR, random.choice(md_list))
print(f'\n===== 抽查文档: {os.path.basename(sample_md)} =====')
with open(sample_md, 'r', encoding='utf-8') as f:
    print(f.read()[:500])

